Objective: Combine the hybrid RAG retrieval system with the fine-tuned ELECTRA duplicate classifier to build an end-to-end support-ticket deflection pipeline.


Step 1: Install Libraries



In [1]:
!pip install transformers sentence-transformers faiss-cpu rank-bm25


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 108.0 MB/s eta 0:00:00


Step 2: Import Libraries

In [2]:
import os
import json
import pickle
import time

import numpy as np
import pandas as pd
import torch
import faiss

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer


Step 1: Upload Files To Colab

In [21]:
from google.colab import files

uploaded = files.upload()


Saving bm25.pkl to bm25.pkl
Saving faiss_index.bin to faiss_index.bin
Saving kb_metadata.csv to kb_metadata.csv
Saving query_eval_set.csv to query_eval_set.csv
Saving rag_config.json to rag_config.json
Saving retrieval_comparison_metrics.csv to retrieval_comparison_metrics.csv


Step 2: Create A Local RAG Folder

In [22]:
import os
import shutil

rag_dir = "/content/rag_assets"
os.makedirs(rag_dir, exist_ok=True)

for filename in uploaded.keys():
    shutil.move(filename, f"{rag_dir}/{filename}")

os.listdir(rag_dir)


['faiss_index.bin',
 'rag_config.json',
 'query_eval_set.csv',
 'kb_metadata.csv',
 'bm25.pkl',
 'retrieval_comparison_metrics.csv']

In [24]:
import json
import pickle
import pandas as pd
import faiss

with open(f"{rag_dir}/rag_config.json", "r") as f:
    rag_config = json.load(f)

kb_df = pd.read_csv(f"{rag_dir}/kb_metadata.csv")
query_df = pd.read_csv(f"{rag_dir}/query_eval_set.csv")

kb_index = faiss.read_index(f"{rag_dir}/faiss_index.bin")

with open(f"{rag_dir}/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)

rag_config, kb_df.shape, query_df.shape, kb_index.ntotal


({'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2',
  'vector_index_type': 'FAISS IndexFlatIP',
  'bm25_enabled': True,
  'hybrid_enabled': True,
  'default_top_k': 5,
  'hybrid_candidate_k': 20,
  'hybrid_alpha': 0.7,
  'metadata_file': 'kb_metadata.csv',
  'faiss_index_file': 'faiss_index.bin',
  'bm25_file': 'bm25.pkl'},
 (21491, 5),
 (5381, 5),
 21491)

 Load Embedding Model

In [25]:
from sentence_transformers import SentenceTransformer

embedding_model_name = rag_config["embedding_model_name"]
embedding_model = SentenceTransformer(embedding_model_name)

embedding_model_name


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'sentence-transformers/all-MiniLM-L6-v2'

 Load Final ELECTRA Classifier

In [26]:
classifier_model_path = "/content/drive/MyDrive/SupportAI/models/final_duplicate_classifier"


In [27]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_model_path)
classifier_model = AutoModelForSequenceClassification.from_pretrained(classifier_model_path)

classifier_model.eval()

print("Classifier loaded successfully")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifier loaded successfully


In [28]:
def retrieve_hybrid_results(query, top_k=5, candidate_k=20, alpha=0.7):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    vector_scores, vector_indices = kb_index.search(query_embedding, candidate_k)

    vector_df = pd.DataFrame({
        "retrieved_index": vector_indices[0],
        "vector_score": vector_scores[0]
    })

    tokenized_query = query.lower().split()
    bm25_scores_all = bm25.get_scores(tokenized_query)
    bm25_indices = np.argsort(bm25_scores_all)[::-1][:candidate_k]

    bm25_df = pd.DataFrame({
        "retrieved_index": bm25_indices,
        "bm25_score": bm25_scores_all[bm25_indices]
    })

    hybrid_df = pd.merge(
        vector_df,
        bm25_df,
        on="retrieved_index",
        how="outer"
    ).fillna(0)

    if hybrid_df["vector_score"].max() > hybrid_df["vector_score"].min():
        hybrid_df["vector_score_norm"] = (
            (hybrid_df["vector_score"] - hybrid_df["vector_score"].min()) /
            (hybrid_df["vector_score"].max() - hybrid_df["vector_score"].min())
        )
    else:
        hybrid_df["vector_score_norm"] = 0

    if hybrid_df["bm25_score"].max() > hybrid_df["bm25_score"].min():
        hybrid_df["bm25_score_norm"] = (
            (hybrid_df["bm25_score"] - hybrid_df["bm25_score"].min()) /
            (hybrid_df["bm25_score"].max() - hybrid_df["bm25_score"].min())
        )
    else:
        hybrid_df["bm25_score_norm"] = 0

    hybrid_df["hybrid_score"] = (
        alpha * hybrid_df["vector_score_norm"] +
        (1 - alpha) * hybrid_df["bm25_score_norm"]
    )

    hybrid_df = hybrid_df.sort_values(
        by="hybrid_score",
        ascending=False
    ).head(top_k)

    results = kb_df.iloc[hybrid_df["retrieved_index"].astype(int)].copy()
    results["retrieved_index"] = hybrid_df["retrieved_index"].values
    results["vector_score"] = hybrid_df["vector_score"].values
    results["bm25_score"] = hybrid_df["bm25_score"].values
    results["hybrid_score"] = hybrid_df["hybrid_score"].values
    results["rank"] = range(1, len(results) + 1)

    return results[[
        "rank",
        "hybrid_score",
        "vector_score",
        "bm25_score",
        "instruction",
        "category",
        "intent",
        "response"
    ]]


In [29]:
retrieve_hybrid_results(
    "I forgot my password and cannot log in",
    top_k=5,
    candidate_k=20,
    alpha=0.7
)


,rank,hybrid_score,vector_score,bm25_score,instruction,category,intent,response
16276,1,0.975731,0.730741,20.261762,"I forgot the fucking pass of my account, I nee...",ACCOUNT,recover_password,I've grasped that you're frustrated and concer...
16520,2,0.972843,0.735767,19.717591,"I forgot my account password, how t reset it?",ACCOUNT,recover_password,"Oh, I completely understand how frustrating it..."
16033,3,0.922687,0.732170,16.388365,I need assistance to reset my user profile pas...,ACCOUNT,recover_password,For sure! I understand that you need assistanc...
16584,4,0.700000,0.740060,0.000000,i cant retrieve my user pwd,ACCOUNT,recover_password,I'm sorry to hear that you're having trouble r...
16027,5,0.696068,0.735903,0.000000,need assistance to recover my account pass,ACCOUNT,recover_password,I'll do my best! I'm here to provide you with ...


Create ELECTRA Pair Scoring Function

In [30]:
def score_duplicate_probability(q1, q2):
    inputs = classifier_tokenizer(
        q1,
        q2,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = classifier_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)
        duplicate_probability = probabilities[0][1].item()

    return duplicate_probability


 Rerank Retrieved Results With ELECTRA

In [31]:
def rerank_with_classifier(query, retrieved_df):
    reranked_df = retrieved_df.copy()

    duplicate_scores = []

    for _, row in reranked_df.iterrows():
        candidate_instruction = row["instruction"]

        score = score_duplicate_probability(
            query,
            candidate_instruction
        )

        duplicate_scores.append(score)

    reranked_df["duplicate_probability"] = duplicate_scores

    reranked_df = reranked_df.sort_values(
        by="duplicate_probability",
        ascending=False
    ).reset_index(drop=True)

    reranked_df["final_rank"] = range(1, len(reranked_df) + 1)

    return reranked_df[[
        "final_rank",
        "duplicate_probability",
        "rank",
        "hybrid_score",
        "instruction",
        "category",
        "intent",
        "response"
    ]]


Step 7: Test Reranking



In [32]:
query = "I forgot my password and cannot log in"

retrieved = retrieve_hybrid_results(query, top_k=5, candidate_k=20, alpha=0.7)

reranked = rerank_with_classifier(query, retrieved)

reranked


,final_rank,duplicate_probability,rank,hybrid_score,instruction,category,intent,response
0,1,0.915907,2,0.972843,"I forgot my account password, how t reset it?",ACCOUNT,recover_password,"Oh, I completely understand how frustrating it..."
1,2,0.865225,1,0.975731,"I forgot the fucking pass of my account, I nee...",ACCOUNT,recover_password,I've grasped that you're frustrated and concer...
2,3,0.697964,3,0.922687,I need assistance to reset my user profile pas...,ACCOUNT,recover_password,For sure! I understand that you need assistanc...
3,4,0.413754,5,0.696068,need assistance to recover my account pass,ACCOUNT,recover_password,I'll do my best! I'm here to provide you with ...
4,5,0.117957,4,0.700000,i cant retrieve my user pwd,ACCOUNT,recover_password,I'm sorry to hear that you're having trouble r...


Create Final SupportAI Function

In [68]:
def supportai_answer(
    query,
    retrieval_top_k=5,
    candidate_k=20,
    alpha=0.7,
    confident_threshold=0.50,
    possible_threshold=0.10
):

    start_time = time.time()

    retrieved = retrieve_hybrid_results(
        query,
        top_k=retrieval_top_k,
        candidate_k=candidate_k,
        alpha=alpha
    )

    reranked = rerank_with_classifier(query, retrieved)

    best_match = reranked.iloc[0]

    total_latency_ms = (time.time() - start_time) * 1000

    if best_match["duplicate_probability"] >= confident_threshold:
        decision = "answer"
        message = "Confident match found."
        final_response = best_match["response"]

    elif best_match["duplicate_probability"] >= possible_threshold:
        decision = "suggest"
        message = "Possible match found. Show top suggestions."
        final_response = best_match["response"]

    else:
        decision = "escalate"
        message = "No confident match found. Escalate to human support."
        final_response = None

    return {
        "query": query,
        "decision": decision,
        "message": message,
        "matched_intent": best_match["intent"],
        "matched_category": best_match["category"],
        "duplicate_probability": best_match["duplicate_probability"],
        "response": final_response,
        "latency_ms": total_latency_ms,
        "top_matches": reranked.head(3)
    }


Test Full Pipeline

In [34]:
result = supportai_answer(
    "I forgot my password and cannot log in"
)

result["decision"], result["matched_intent"], result["duplicate_probability"], result["response"]


('answer',
 'recover_password',
 np.float64(0.9159073829650879),
 'Oh, I completely understand how frustrating it can be to forget your account password. Don\'t worry, I\'m here to assist you in the process of resetting it so you can regain access to your account. Here\'s what you can do:\n\n1. First and foremost, visit our login page at "{{Login Page URL}}" to initiate the password reset process.\n2. Look for the option that says "{{Forgot Password}}" and click on it.\n3. You will be prompted to enter the email address associated with your account. Please provide the correct email address to proceed.\n4. Once you\'ve entered your email address, an email will be sent to you shortly with instructions on how to reset your password. Please check your inbox, as well as your spam or junk folder, just in case.\n5. Follow the instructions in the email to create a new password. Make sure to choose a strong and unique password to enhance the security of your account.\n6. After resetting your pa

In [35]:
result["top_matches"]


,final_rank,duplicate_probability,rank,hybrid_score,instruction,category,intent,response
0,1,0.915907,2,0.972843,"I forgot my account password, how t reset it?",ACCOUNT,recover_password,"Oh, I completely understand how frustrating it..."
1,2,0.865225,1,0.975731,"I forgot the fucking pass of my account, I nee...",ACCOUNT,recover_password,I've grasped that you're frustrated and concer...
2,3,0.697964,3,0.922687,I need assistance to reset my user profile pas...,ACCOUNT,recover_password,For sure! I understand that you need assistanc...


In [36]:
supportai_answer("I want to cancel my order")


{'query': 'I want to cancel my order',
 'decision': 'answer',
 'message': 'Confident match found.',
 'matched_intent': 'cancel_order',
 'matched_category': 'ORDER',
 'duplicate_probability': np.float64(0.9371426105499268),
 'response': "I've realized that you're seeking assistance in canceling an order. I'll be happy to guide you through the process. Could you please provide me with the order number or any relevant details about the order you would like to cancel? With that information, I'll be able to provide you with the specific steps to cancel your order successfully.",
 'latency_ms': 315.3665065765381,
 'top_matches':    final_rank  duplicate_probability  rank  hybrid_score  \
 0           1               0.937143     5      0.685626   
 1           2               0.746159     3      0.928120   
 2           3               0.342306     2      0.931442   
 
                                          instruction category        intent  \
 0                                       can

In [37]:
supportai_answer("I was charged twice and need a refund")


{'query': 'I was charged twice and need a refund',
 'decision': 'escalate',
 'message': 'No confident match found. Escalate to human support.',
 'matched_intent': 'check_refund_policy',
 'matched_category': 'REFUND',
 'duplicate_probability': np.float64(0.09600596874952316),
 'response': None,
 'latency_ms': 378.8726329803467,
 'top_matches':    final_rank  duplicate_probability  rank  hybrid_score  \
 0           1               0.096006     2      0.693422   
 1           2               0.007148     1      0.700000   
 2           3               0.002499     4      0.686408   
 
                                          instruction category  \
 0   i have gotto check ur damn refund policy help me   REFUND   
 1   under which circumstances can I request refunds?   REFUND   
 2  i have to see in what cases acn i request my m...   REFUND   
 
                 intent                                           response  
 0  check_refund_policy  Definitely! I completely understand your u

In [38]:
supportai_answer("My delivery address is wrong")


{'query': 'My delivery address is wrong',
 'decision': 'answer',
 'message': 'Confident match found.',
 'matched_intent': 'change_shipping_address',
 'matched_category': 'SHIPPING',
 'duplicate_probability': np.float64(0.9664813876152039),
 'response': 'We apologize for any inconvenience caused by the incorrect delivery address. To assist you in correcting it, please provide us with the correct address details, including the street, city, state, and postal code. Once we have the accurate information, we will promptly update your delivery address and ensure your package is sent to the correct location. Thank you for your understanding and cooperation.',
 'latency_ms': 348.5708236694336,
 'top_matches':    final_rank  duplicate_probability  rank  hybrid_score  \
 0           1               0.966481     1      0.958553   
 1           2               0.951006     2      0.957378   
 2           3               0.942672     3      0.945526   
 
                                          in

Pretty Print Function

In [39]:
def print_supportai_result(result):
    print("Query:")
    print(result["query"])

    print("\nDecision:")
    print(result["decision"])

    print("\nMatched Intent:")
    print(result["matched_intent"])

    print("\nMatched Category:")
    print(result["matched_category"])

    print("\nDuplicate Probability:")
    print(round(float(result["duplicate_probability"]), 4))

    print("\nResponse:")
    if result["response"] is not None:
        print(result["response"])
    else:
        print("No confident response found. Escalate to human support.")

    print("\nTop Matches:")
    display(result["top_matches"])


In [40]:
result = supportai_answer("I was charged twice and need a refund")
print_supportai_result(result)


Query:
I was charged twice and need a refund

Decision:
escalate

Matched Intent:
check_refund_policy

Matched Category:
REFUND

Duplicate Probability:
0.096

Response:
No confident response found. Escalate to human support.

Top Matches:


,final_rank,duplicate_probability,rank,hybrid_score,instruction,category,intent,response
0,1,0.096006,2,0.693422,i have gotto check ur damn refund policy help me,REFUND,check_refund_policy,Definitely! I completely understand your urgen...
1,2,0.007148,1,0.700000,under which circumstances can I request refunds?,REFUND,check_refund_policy,Unquestionably! I'm here to provide you with a...
2,3,0.002499,4,0.686408,i have to see in what cases acn i request my m...,REFUND,check_refund_policy,I'm happy to help! I completely understand you...


In the charged twice refund example, the retrieval system found refund policy records, but the ELECTRA duplicate classifier assigned a low duplicate probability. The system therefore escalated the query instead of returning a loosely related answer. This demonstrates confidence based deflection: the system only answers when it has a strong same intent match.


End-To-End Evaluation Function

In [41]:
def evaluate_end_to_end_pipeline(query_df, sample_size=500):
    eval_sample = query_df.sample(
        n=min(sample_size, len(query_df)),
        random_state=42
    )

    records = []
    latencies = []

    for _, row in eval_sample.iterrows():
        query = row["instruction"]
        true_intent = row["intent"]
        true_category = row["category"]

        result = supportai_answer(query)
        latencies.append(result["latency_ms"])

        matched_intent = result["matched_intent"]
        matched_category = result["matched_category"]

        records.append({
            "query": query,
            "true_intent": true_intent,
            "matched_intent": matched_intent,
            "true_category": true_category,
            "matched_category": matched_category,
            "decision": result["decision"],
            "duplicate_probability": float(result["duplicate_probability"]),
            "intent_correct": int(true_intent == matched_intent),
            "category_correct": int(true_category == matched_category),
            "response_returned": result["response"] is not None,
            "latency_ms": result["latency_ms"]
        })

    eval_results_df = pd.DataFrame(records)

    metrics = {
        "intent_accuracy": eval_results_df["intent_correct"].mean(),
        "category_accuracy": eval_results_df["category_correct"].mean(),
        "answer_rate": (eval_results_df["decision"] == "answer").mean(),
        "suggest_rate": (eval_results_df["decision"] == "suggest").mean(),
        "escalation_rate": (eval_results_df["decision"] == "escalate").mean(),
        "avg_duplicate_probability": eval_results_df["duplicate_probability"].mean(),
        "avg_latency_ms": np.mean(latencies),
        "p95_latency_ms": np.percentile(latencies, 95)
    }

    return metrics, eval_results_df


In [42]:
end_to_end_metrics, end_to_end_eval_df = evaluate_end_to_end_pipeline(
    query_df,
    sample_size=500
)

end_to_end_metrics


{'intent_accuracy': np.float64(0.972),
 'category_accuracy': np.float64(1.0),
 'answer_rate': np.float64(0.888),
 'suggest_rate': np.float64(0.06),
 'escalation_rate': np.float64(0.052),
 'avg_duplicate_probability': np.float64(0.9001971392203122),
 'avg_latency_ms': np.float64(330.36565685272217),
 'p95_latency_ms': np.float64(416.9946551322936)}

In [43]:
end_to_end_eval_df.to_csv(
    "/content/drive/MyDrive/SupportAI/end_to_end_eval_results.csv",
    index=False
)

pd.DataFrame([end_to_end_metrics]).to_csv(
    "/content/drive/MyDrive/SupportAI/end_to_end_metrics.csv",
    index=False
)


Create Custom Validation DataFrame

In [58]:
custom_validation_data = [
    {
        "query": "I forgot my password and cannot access my account.",
        "expected_category": "ACCOUNT",
        "expected_intent": "recover_password",
        "expected_decision": "answer"
    },
    {
        "query": "I need to reset my login password.",
        "expected_category": "ACCOUNT",
        "expected_intent": "recover_password",
        "expected_decision": "answer"
    },
    {
        "query": "My delivery address is wrong. Can I update it?",
        "expected_category": "SHIPPING",
        "expected_intent": "change_shipping_address",
        "expected_decision": "answer"
    },
    {
        "query": "The package is going to the wrong address.",
        "expected_category": "SHIPPING",
        "expected_intent": "change_shipping_address",
        "expected_decision": "answer"
    },
    {
        "query": "I want to cancel my order before it ships.",
        "expected_category": "ORDER",
        "expected_intent": "cancel_order",
        "expected_decision": "answer"
    },
    {
        "query": "Please cancel the item I bought yesterday.",
        "expected_category": "ORDER",
        "expected_intent": "cancel_order",
        "expected_decision": "answer"
    },
    {
        "query": "Where is my refund?",
        "expected_category": "REFUND",
        "expected_intent": "track_refund",
        "expected_decision": "answer"
    },
    {
        "query": "Can you tell me the status of my refund?",
        "expected_category": "REFUND",
        "expected_intent": "track_refund",
        "expected_decision": "answer"
    },
    {
        "query": "What is your refund policy?",
        "expected_category": "REFUND",
        "expected_intent": "check_refund_policy",
        "expected_decision": "answer"
    },
    {
        "query": "When am I allowed to request a refund?",
        "expected_category": "REFUND",
        "expected_intent": "check_refund_policy",
        "expected_decision": "answer"
    },
    {
        "query": "I was charged twice and need my money back.",
        "expected_category": "REFUND",
        "expected_intent": "check_refund_policy",
        "expected_decision": "suggest"
    },
    {
        "query": "I paid two times for the same order.",
        "expected_category": "REFUND",
        "expected_intent": "check_refund_policy",
        "expected_decision": "suggest"
    },
    {
        "query": "I want to change the email address on my account.",
        "expected_category": "ACCOUNT",
        "expected_intent": "edit_account",
        "expected_decision": "answer"
    },
    {
        "query": "How do I update my account email?",
        "expected_category": "ACCOUNT",
        "expected_intent": "edit_account",
        "expected_decision": "answer"
    },
    {
        "query": "My order never arrived.",
        "expected_category": "SHIPPING",
        "expected_intent": "track_order",
        "expected_decision": "suggest"
    },
    {
        "query": "Can you help me find where my package is?",
        "expected_category": "SHIPPING",
        "expected_intent": "track_order",
        "expected_decision": "answer"
    },
    {
        "query": "I want to delete my account permanently.",
        "expected_category": "ACCOUNT",
        "expected_intent": "delete_account",
        "expected_decision": "answer"
    },
    {
        "query": "Please remove my profile and all my data.",
        "expected_category": "ACCOUNT",
        "expected_intent": "delete_account",
        "expected_decision": "answer"
    },
    {
        "query": "The app keeps crashing every time I open it.",
        "expected_category": "UNKNOWN",
        "expected_intent": "UNKNOWN",
        "expected_decision": "escalate"
    },
    {
        "query": "I need help with something not listed here.",
        "expected_category": "UNKNOWN",
        "expected_intent": "UNKNOWN",
        "expected_decision": "escalate"
    }
]

custom_validation_df = pd.DataFrame(custom_validation_data)
custom_validation_df


,query,expected_category,expected_intent,expected_decision
0,I forgot my password and cannot access my acco...,ACCOUNT,recover_password,answer
1,I need to reset my login password.,ACCOUNT,recover_password,answer
2,My delivery address is wrong. Can I update it?,SHIPPING,change_shipping_address,answer
3,The package is going to the wrong address.,SHIPPING,change_shipping_address,answer
4,I want to cancel my order before it ships.,ORDER,cancel_order,answer
5,Please cancel the item I bought yesterday.,ORDER,cancel_order,answer
6,Where is my refund?,REFUND,track_refund,answer
7,Can you tell me the status of my refund?,REFUND,track_refund,answer
8,What is your refund policy?,REFUND,check_refund_policy,answer
9,When am I allowed to request a refund?,REFUND,check_refund_policy,answer


Run SupportAI On Each Query

In [69]:
custom_results = []

for _, row in custom_validation_df.iterrows():
    result = supportai_answer(row["query"])

    custom_results.append({
        "query": row["query"],
        "expected_category": row["expected_category"],
        "predicted_category": result["matched_category"],
        "expected_intent": row["expected_intent"],
        "predicted_intent": result["matched_intent"],
        "expected_decision": row["expected_decision"],
        "predicted_decision": result["decision"],
        "duplicate_probability": float(result["duplicate_probability"]),
        "category_correct": int(row["expected_category"] == result["matched_category"]),
        "intent_correct": int(row["expected_intent"] == result["matched_intent"]),
        "decision_correct": int(row["expected_decision"] == result["decision"]),
        "latency_ms": result["latency_ms"]
    })

custom_results_df = pd.DataFrame(custom_results)
custom_results_df


,query,expected_category,predicted_category,expected_intent,predicted_intent,expected_decision,predicted_decision,duplicate_probability,category_correct,intent_correct,decision_correct,latency_ms
0,I forgot my password and cannot access my acco...,ACCOUNT,ACCOUNT,recover_password,recover_password,answer,answer,0.971378,1,1,1,458.321810
1,I need to reset my login password.,ACCOUNT,ACCOUNT,recover_password,recover_password,answer,answer,0.980021,1,1,1,444.559813
2,My delivery address is wrong. Can I update it?,SHIPPING,SHIPPING,change_shipping_address,change_shipping_address,answer,answer,0.982969,1,1,1,447.428942
3,The package is going to the wrong address.,SHIPPING,SHIPPING,change_shipping_address,change_shipping_address,answer,suggest,0.320219,1,1,0,338.301659
4,I want to cancel my order before it ships.,ORDER,ORDER,cancel_order,cancel_order,answer,answer,0.964654,1,1,1,310.582876
5,Please cancel the item I bought yesterday.,ORDER,ORDER,cancel_order,cancel_order,answer,escalate,0.032893,1,1,0,305.254698
6,Where is my refund?,REFUND,REFUND,track_refund,get_refund,answer,answer,0.973438,1,0,1,275.572777
7,Can you tell me the status of my refund?,REFUND,REFUND,track_refund,track_refund,answer,answer,0.938694,1,1,1,337.573290
8,What is your refund policy?,REFUND,REFUND,check_refund_policy,check_refund_policy,answer,answer,0.958717,1,1,1,289.554596
9,When am I allowed to request a refund?,REFUND,REFUND,check_refund_policy,get_refund,answer,answer,0.980256,1,0,1,315.788746


 Calculate Custom Metrics

In [70]:
known_cases = custom_results_df[custom_results_df["expected_intent"] != "UNKNOWN"]

custom_metrics = {
    "known_category_accuracy": known_cases["category_correct"].mean(),
    "known_intent_accuracy": known_cases["intent_correct"].mean(),
    "decision_accuracy": custom_results_df["decision_correct"].mean(),
    "avg_duplicate_probability": custom_results_df["duplicate_probability"].mean(),
    "avg_latency_ms": custom_results_df["latency_ms"].mean(),
    "p95_latency_ms": custom_results_df["latency_ms"].quantile(0.95),
    "num_queries": len(custom_results_df)
}

custom_metrics


{'known_category_accuracy': np.float64(0.7777777777777778),
 'known_intent_accuracy': np.float64(0.6666666666666666),
 'decision_accuracy': np.float64(0.6),
 'avg_duplicate_probability': np.float64(0.5177201178652467),
 'avg_latency_ms': np.float64(329.21828031539917),
 'p95_latency_ms': np.float64(447.9735851287842),
 'num_queries': 20}

In [61]:
failed_cases = custom_results_df[
    (custom_results_df["intent_correct"] == 0) |
    (custom_results_df["decision_correct"] == 0)
]

failed_cases[[
    "query",
    "expected_intent",
    "predicted_intent",
    "expected_decision",
    "predicted_decision",
    "duplicate_probability"
]]


,query,expected_intent,predicted_intent,expected_decision,predicted_decision,duplicate_probability
3,The package is going to the wrong address.,change_shipping_address,change_shipping_address,answer,escalate,0.320219
5,Please cancel the item I bought yesterday.,cancel_order,cancel_order,answer,escalate,0.032893
6,Where is my refund?,track_refund,get_refund,answer,answer,0.973438
9,When am I allowed to request a refund?,check_refund_policy,get_refund,answer,answer,0.980256
10,I was charged twice and need my money back.,check_refund_policy,get_refund,suggest,suggest,0.610443
11,I paid two times for the same order.,check_refund_policy,cancel_order,suggest,escalate,0.007645
12,I want to change the email address on my account.,edit_account,change_shipping_address,answer,escalate,0.120853
13,How do I update my account email?,edit_account,edit_account,answer,suggest,0.567156
14,My order never arrived.,track_order,track_order,suggest,escalate,0.002578
15,Can you help me find where my package is?,track_order,delivery_period,answer,escalate,0.110078


In [53]:
kb_df["intent"].sort_values().unique()


array(['cancel_order', 'change_order', 'change_shipping_address',
       'check_cancellation_fee', 'check_invoice', 'check_payment_methods',
       'check_refund_policy', 'complaint', 'contact_customer_service',
       'contact_human_agent', 'create_account', 'delete_account',
       'delivery_options', 'delivery_period', 'edit_account',
       'get_invoice', 'get_refund', 'newsletter_subscription',
       'payment_issue', 'place_order', 'recover_password',
       'registration_problems', 'review', 'set_up_shipping_address',
       'switch_account', 'track_order', 'track_refund'], dtype=object)

In [54]:
kb_df[kb_df["category"] == "REFUND"]["intent"].value_counts()


,count
intent,
track_refund,798
check_refund_policy,797
get_refund,797


In [55]:
kb_df[kb_df["category"] == "ACCOUNT"]["intent"].value_counts()


,count
intent,
switch_account,800
edit_account,800
registration_problems,799
create_account,797
delete_account,796
recover_password,796


In [56]:
kb_df[kb_df["category"] == "SHIPPING"]["intent"].value_counts()


,count
intent,
set_up_shipping_address,797
change_shipping_address,778


In [57]:
kb_df[kb_df["category"] == "ORDER"]["intent"].value_counts()


,count
intent,
cancel_order,798
place_order,798
change_order,797
track_order,796


In [67]:
def assign_decision(prob, answer_threshold, suggest_threshold):
    if prob >= answer_threshold:
        return "answer"
    elif prob >= suggest_threshold:
        return "suggest"
    else:
        return "escalate"


threshold_results = []

answer_thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
suggest_thresholds = [0.10, 0.20, 0.25, 0.30, 0.40, 0.50]

for answer_t in answer_thresholds:
    for suggest_t in suggest_thresholds:
        if suggest_t >= answer_t:
            continue

        temp_df = custom_results_df.copy()

        temp_df["threshold_decision"] = temp_df["duplicate_probability"].apply(
            lambda p: assign_decision(p, answer_t, suggest_t)
        )

        temp_df["threshold_decision_correct"] = (
            temp_df["expected_decision"] == temp_df["threshold_decision"]
        ).astype(int)

        false_deflection_rate = (
            (temp_df["expected_decision"] == "escalate") &
            (temp_df["threshold_decision"] == "answer")
        ).mean()

        over_escalation_rate = (
            (temp_df["expected_decision"].isin(["answer", "suggest"])) &
            (temp_df["threshold_decision"] == "escalate")
        ).mean()

        threshold_results.append({
            "answer_threshold": answer_t,
            "suggest_threshold": suggest_t,
            "decision_accuracy": temp_df["threshold_decision_correct"].mean(),
            "false_deflection_rate": false_deflection_rate,
            "over_escalation_rate": over_escalation_rate
        })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.sort_values(
    by=["false_deflection_rate", "decision_accuracy"],
    ascending=[True, False]
).head(20)


,answer_threshold,suggest_threshold,decision_accuracy,false_deflection_rate,over_escalation_rate
0,0.50,0.10,0.60,0.0,0.15
1,0.50,0.20,0.60,0.0,0.25
2,0.50,0.25,0.60,0.0,0.25
3,0.50,0.30,0.60,0.0,0.30
4,0.50,0.40,0.60,0.0,0.35
5,0.55,0.10,0.60,0.0,0.15
6,0.55,0.20,0.60,0.0,0.25
7,0.55,0.25,0.60,0.0,0.25
8,0.55,0.30,0.60,0.0,0.30
9,0.55,0.40,0.60,0.0,0.35


In [71]:
custom_results_df["false_deflection"] = (
    (custom_results_df["expected_decision"] == "escalate") &
    (custom_results_df["predicted_decision"] == "answer")
).astype(int)

custom_results_df["over_escalation"] = (
    (custom_results_df["expected_decision"].isin(["answer", "suggest"])) &
    (custom_results_df["predicted_decision"] == "escalate")
).astype(int)

business_metrics = {
    **custom_metrics,
    "false_deflection_rate": custom_results_df["false_deflection"].mean(),
    "over_escalation_rate": custom_results_df["over_escalation"].mean(),
    "answer_threshold": 0.50,
    "suggest_threshold": 0.10
}

business_metrics


{'known_category_accuracy': np.float64(0.7777777777777778),
 'known_intent_accuracy': np.float64(0.6666666666666666),
 'decision_accuracy': np.float64(0.6),
 'avg_duplicate_probability': np.float64(0.5177201178652467),
 'avg_latency_ms': np.float64(329.21828031539917),
 'p95_latency_ms': np.float64(447.9735851287842),
 'num_queries': 20,
 'false_deflection_rate': np.float64(0.0),
 'over_escalation_rate': np.float64(0.15),
 'answer_threshold': 0.5,
 'suggest_threshold': 0.1}